# Lets Ingest JSON based parsing using langchain  

# LangChain JSONLoader Overview

The `JSONLoader` in LangChain extracts documents from JSON or JSON Lines (`.jsonl`) files by leveraging a `jq` schema to parse specific keys or items into standard Document objects.

## Key Features
* **JQ Schema Integration:** Uses `jq` syntax to pinpoint and extract specific arrays or nested fields without loading the entire unformatted file into memory.
* **Metadata Extraction:** Seamlessly pulls metadata from surrounding JSON nodes and attaches it to each generated document.
* **JSONL Support:** Easily handles line-delimited JSON files via configuration flags or standard jq selectors.

## Basic Usage Example
```python
from langchain_community.document_loaders import JSONLoader

loader = JSONLoader(
    file_path="data.json",
    jq_schema=".messages[].content",
    text_content=False
)

documents = loader.load()
```

## Pros & Cons
* **Pros:** Highly precise extraction; avoids noise from irrelevant nested JSON structures; powerful jq filtering capabilities.
* **Cons:** Requires the external `jq` Python package and binary installation; strict schema requirements can break if the JSON structure changes unexpectedly.

In [ ]:
import os
import json
os.makedirs("data/json_file",exist_ok=True)

In [3]:
sample_Json = {
    "company": "TechCorp",
    "employees": [
            {
                "id": 1,
                "name": "John Doe",
                "role": "Software Engineer",
                "skills": ["Python", "JavaScript", "React"],
                "projects": [
                    {"name": "RAG System", "status": "In Progress"},
                    {"name": "Data Pipeline", "status": "Completed"}
                ]
            },
            {
             "id": 2,
             "name": "Jane Smith",
             "role": "Data Scientist",
             "skills": ["Python", "Machine Learning", "SQL"],
             "projects": [
             {"name": "ML Model", "status": "In Progress"},
             {"name": "Analytics Dashboard", "status": "Planning"}
             ]
             }
             ],
            "departments": {
                "engineering": {
                "head": "Mike Johnson",
                "budget": 1000000,
                "team_size": 25
             },
             "data_science": {
               "head": "Sarah Williams",
               "budget": 750000,
               "team_size": 15
               }
            }
}

In [11]:
sample_Json

{'company': 'TechCorp',
 'employees': [{'id': 1,
   'name': 'John Doe',
   'role': 'Software Engineer',
   'skills': ['Python', 'JavaScript', 'React'],
   'projects': [{'name': 'RAG System', 'status': 'In Progress'},
    {'name': 'Data Pipeline', 'status': 'Completed'}]},
  {'id': 2,
   'name': 'Jane Smith',
   'role': 'Data Scientist',
   'skills': ['Python', 'Machine Learning', 'SQL'],
   'projects': [{'name': 'ML Model', 'status': 'In Progress'},
    {'name': 'Analytics Dashboard', 'status': 'Planning'}]}],
 'departments': {'engineering': {'head': 'Mike Johnson',
   'budget': 1000000,
   'team_size': 25},
  'data_science': {'head': 'Sarah Williams',
   'budget': 750000,
   'team_size': 15}}}

In [17]:
with open('data/json_file/sample_data.json','w') as j:
    json.dump(sample_Json,j,indent=2)

In [18]:
# Save in JSON Lines format
jsonl_data = [
    {"timestamp": "2024-01-01", "event": "user_login", "user_id": 123},
    {"timestamp": "2024-01-01", "event": "page_view", "user_id": 123, "page": "/home"},
    {"timestamp": "2024-01-01", "event": "purchase", "user_id": 853, "amount": 89},
    {"timestamp": "2024-01-05", "event": "page_view", "user_id": 853, "page": "/home"},
    {"timestamp": "2024-01-02", "event": "purchase", "user_id": 183, "amount": 9},
    {"timestamp": "2024-01-03", "event": "purchase", "user_id": 163, "amount": 99}
]

with open('data/json_file/events.jsonl', 'w') as f:
    for item in jsonl_data:
        f.write(json.dumps(item) + '\n')

# Parsing using JSONLoader

In [7]:
## MEthod1 : JsonLoader With jq_schema
print("1️⃣ JSONLoader - Extract specific fields")
from langchain_community.document_loaders import JSONLoader
# Extract employee information
emp_loader = JSONLoader(
    file_path='data/json_file/sample_data.json',
    jq_schema='.employees[]',  # jq query to extract each employee
    text_content=False  # Get full JSON objects
)

employee_docs = emp_loader.load()
print(f"Loaded {len(employee_docs)} employee documents")
print(f"First employee: {employee_docs[0].page_content[:200]}...")
print(employee_docs)

1️⃣ JSONLoader - Extract specific fields
Loaded 2 employee documents
First employee: {"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status"...
[Document(metadata={'source': 'D:\\Rag Vyuham\\DataIngestParsing\\data\\json_file\\sample_data.json', 'seq_num': 1}, page_content='{"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status": "Completed"}]}'), Document(metadata={'source': 'D:\\Rag Vyuham\\DataIngestParsing\\data\\json_file\\sample_data.json', 'seq_num': 2}, page_content='{"id": 2, "name": "Jane Smith", "role": "Data Scientist", "skills": ["Python", "Machine Learning", "SQL"], "projects": [{"name": "ML Model", "status": "In Progress"}, {"name": "Analytics Dashboard", "status": "Planning"}]}')]


In [24]:
from typing import List,Dict
from langchain_core.documents import Document
import json
def cust_JSON_Load(filepath:str)-> List[Document]:
    """Process JSON with intelligent flattening and context preservation"""
    with open(filepath) as f:
        data = json.load(f)
    docs = []

    # Strategy 1: Create documents for each employee with full context
    for emp in data.get('employees',[]):
        content = f"""Employee Profile:
        Name:{emp['name']} 
        Role: {emp['role']} 
        skills:{','.join(emp['skills'])}
        Projects:"""
        for prj in emp.get('projects',[]):
            content += f"\n\t - {prj['name']} (Status: {prj['status']})"
        doc = Document(
            page_content= content,
            metadata = {
                'source': filepath,
                'data_type': 'employee_profile',
                'employee_id': emp['id'],
                'employee_name': emp['name'],
                'role': emp['role']
            }
        )
        docs.append(doc)
    return docs

In [ ]:
print(cust_JSON_Load)
try:
    dataa = cust_JSON_Load("data/json_file/sample_data.json") # try this path: 'data/json_file/events.jsonl'
    print(f'Number of emplotees:',len(dataa))
    for i in range(len(dataa)):
        print(f'Employee: {i+1} Roles and Res are: \n',dataa[i].page_content)
        print("+++"*11)
        print(f'Employee: {i+1} Metadata: \n',dataa[i].metadata)
        print("***"*21)
except Exception as e:
    print("uh oww error occured:",e)

<function cust_JSON_Load at 0x000001A3C88D5260>
uh oww error occured: Extra data: line 2 column 1 (char 67)


# Jsonl Loader based CustomFunction


# Functional Differences: JSON vs. JSONL in Custom Loaders

Using the custom function to read JSONL type format requires shifting from an in-memory document tree paradigm to a streaming, line-by-line record paradigm. Below is the functional breakdown of how this impacts your custom parsing logic.

## 1. Structural & Parsing Mechanism

* **Standard JSON (`json.load`):**
  * **Behavior:** Reads the entire file as a single hierarchical data structure (typically a root dictionary containing arrays).
  * **Memory Impact:** Loads the entire file into memory at once. If the JSON file grows to gigabytes, it can cause memory spikes.
  * **Error Handling:** If there is a syntax error anywhere in the file, the entire `json.load()` call fails completely.

* **JSON Lines (`jsonlines` or `json.loads` per line):**
  * **Behavior:** Treats each line as a completely independent, valid JSON object. 
  * **Memory Impact:** Highly memory-efficient (streaming). You can process massive datasets line-by-line without loading everything into RAM.
  * **Error Handling:** If a single line is malformed, you can catch the exception for that specific line (e.g., using `try...except json.JSONDecodeError` inside the loop) without breaking the ingestion of the rest of the file.

## 2. Impact on Custom Loader Functions

When transitioning your custom parser from JSON to JSONL, the internal traversal logic changes:

* **JSON Traversal:** Requires accessing nested keys from a root object (e.g., navigating `data.get('employees', [])` because all records are nested inside a parent array).
* **JSONL Traversal:** The file itself *is* the array. Each iteration of the loop yields an individual record directly (e.g., each line `emp` is already an individual employee object).

## Summary Table

| Feature | Standard JSON (`json.load`) | JSON Lines (`jsonl` / `jsonlines`) |
| :--- | :--- | :--- |
| **Root Structure** | Must have a single root object or array | Every line is a standalone root object |
| **Parsing Function** | `json.load(f)` | `jsonlines.open(f)` or `json.loads(line)` |
| **Scalability** | Poor for very large files (loads all to RAM) | Excellent (stream-based line reading) |
| **Fault Tolerance** | All-or-nothing parsing failure | Line-by-line exception handling |

In [31]:
from typing import List
from langchain_core.documents import Document
import json

def cust_Event_JSONL_Load(filepath: str) -> List[Document]:
    """Process event-based JSONL files line-by-line into LangChain Documents"""
    docs = []
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
                
            try:
                event = json.loads(line)
                
                # Extract core fields
                timestamp = event.get('timestamp', 'N/A')
                event_type = event.get('event', 'N/A')
                user_id = event.get('user_id', 'N/A')
                
                # Build flexible page content based on event type details
                content = f"Event Log:\n- Timestamp: {timestamp}\n- Type: {event_type}\n- User ID: {user_id}"
                
                if 'page' in event:
                    content += f"\n- Page Visited: {event['page']}"
                if 'amount' in event:
                    content += f"\n- Purchase Amount: ${event['amount']}"
                
                doc = Document(
                    page_content=content,
                    metadata={
                        'source': filepath,
                        'data_type': 'event_log',
                        'timestamp': timestamp,
                        'event': event_type,
                        'user_id': user_id
                    }
                )
                docs.append(doc)
                
            except json.JSONDecodeError as je:
                print(f"Skipping malformed JSON on line {line_num}: {je}")
                
    return docs

# Testing the custom function
try:
    events_docs = cust_Event_JSONL_Load('data/json_file/events.jsonl')
    print(f'Total Events Loaded: {len(events_docs)}')
    for i, doc in enumerate(events_docs[:3]):
        print(f'--- Event {i+1} ---'*3)
        print(doc.page_content)
        print("Metadata:", doc.metadata)
except Exception as e:
    print("uh oww error occurred:", e)

Total Events Loaded: 6
--- Event 1 ------ Event 1 ------ Event 1 ---
Event Log:
- Timestamp: 2024-01-01
- Type: user_login
- User ID: 123
Metadata: {'source': 'data/json_file/events.jsonl', 'data_type': 'event_log', 'timestamp': '2024-01-01', 'event': 'user_login', 'user_id': 123}
--- Event 2 ------ Event 2 ------ Event 2 ---
Event Log:
- Timestamp: 2024-01-01
- Type: page_view
- User ID: 123
- Page Visited: /home
Metadata: {'source': 'data/json_file/events.jsonl', 'data_type': 'event_log', 'timestamp': '2024-01-01', 'event': 'page_view', 'user_id': 123}
--- Event 3 ------ Event 3 ------ Event 3 ---
Event Log:
- Timestamp: 2024-01-01
- Type: purchase
- User ID: 853
- Purchase Amount: $89
Metadata: {'source': 'data/json_file/events.jsonl', 'data_type': 'event_log', 'timestamp': '2024-01-01', 'event': 'purchase', 'user_id': 853}
